# cjm-capability-ffmpeg

> An FFmpeg-based media-processing capability for the cjm-substrate runtime that provides audio extraction, segmentation, and format conversion.

## Install

```bash
pip install cjm_capability_ffmpeg
```

## Project Structure

```
nbs/
├── utils/ (5)
│   ├── availability.ipynb  # Detect whether the `ffmpeg` binary is installed on this system.
│   ├── codec.ipynb         # Map audio container formats to the ffmpeg codec used to encode them.
│   ├── probe.ipynb         # Probe media files for metadata (duration, ...) via ffprobe.
│   ├── progress.ipynb      # Run ffmpeg subprocess commands with a progress bar and optional callback.
│   └── segments.ipynb      # Extract temporal segments from audio files via ffmpeg stream-copy.
└── capability.ipynb  # FFmpeg-based media processing capability implementing the `MediaProcessingCapability` interface.
```

Total: 6 notebooks across 1 directory

## Module Dependencies

```mermaid
graph LR
    capability["capability<br/>FFmpeg Processing Capability"]
    utils_availability["utils.availability<br/>FFmpeg Availability"]
    utils_codec["utils.codec<br/>Audio Codec Map"]
    utils_probe["utils.probe<br/>Media Probing"]
    utils_progress["utils.progress<br/>FFmpeg Execution + Progress"]
    utils_segments["utils.segments<br/>Audio Segment Extraction"]

    capability --> utils_probe
    capability --> utils_codec
    capability --> utils_progress
    capability --> utils_availability
    capability --> utils_segments
    utils_segments --> utils_progress
```

*6 cross-module dependencies detected*

## CLI Reference

No CLI commands found in this project.

## Module Overview

Detailed documentation for each module in the project:

### FFmpeg Availability (`availability.ipynb`)
> Detect whether the `ffmpeg` binary is installed on this system.

#### Import

```python
from cjm_capability_ffmpeg.utils.availability import (
    FFMPEG_AVAILABLE
)
```
#### Variables

```python
FFMPEG_AVAILABLE
```

### FFmpeg Processing Capability (`capability.ipynb`)
> FFmpeg-based media processing capability implementing the `MediaProcessingCapability` interface.

#### Import

```python
from cjm_capability_ffmpeg.capability import (
    FFmpegCapabilityConfig,
    FFmpegProcessingCapability
)
```
#### Classes

```python
@dataclass
class FFmpegCapabilityConfig:
    """
    Configuration for the FFmpeg processing tool (the HOW knobs only).
    
    Per-call WHAT-to-produce params (output_format / sample_rate / channels /
    boundaries) are method arguments the adapter hashes into the cache key, NOT
    config — only the quality/engine knobs live here. (The fused-era `output_dir`
    config field is gone: the adapter chooses the output location, stage-8 L6.)
    """
    
    default_audio_format: str = field(...)
    default_audio_bitrate: str = field(...)
    prefer_stream_copy: bool = field(...)
    resampler: str = field(...)
```

```python
class FFmpegProcessingCapability:
    def __init__(self):
        """Initialize the FFmpeg processing tool."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Optional[FFmpegCapabilityConfig] = None
    """
    FFmpeg-based media-processing tool capability (stage 8: pure compute).
    
    Native-surface model (PILLAR 1c): PURE COMPUTE. The artifact ops
    (`convert`/`segment_audio`/`extract_audio`) read input, run ffmpeg, and WRITE
    file(s) into the adapter-supplied `output_dir`, returning typed DTOs;
    `get_info` probes and returns `MediaMetadata`. The cache-check +
    output-location choice + persistence live in the generic media-processing
    adapter (cjm-media-processing-adapter-interface); result DTOs live in
    cjm-capability-primitives; identity derives from the installed distribution.
    No `get_plugin_metadata`, no `self.storage`, no in-tool cache, no
    `execute`/`@capability_action` dispatcher (the task channel routes by method).
    
    Config = the HOW knobs (resampler engine, bitrate, stream-copy preference,
    fallback format); the per-call output_format/sample_rate/channels/boundaries
    are WHAT-to-produce request params the adapter hashes into the cache key.
    The fused-era `extract_segment` action was DROPPED (no consumer; the HITL
    audio-chunk read is a future in-memory op, a different shape).
    """
    
    def __init__(self):
            """Initialize the FFmpeg processing tool."""
            self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
            self.config: Optional[FFmpegCapabilityConfig] = None
        "Initialize the FFmpeg processing tool."
    
    def name(self) -> str:  # Tool identity, derived from the installed distribution (PILLAR 1c)
            from importlib.metadata import metadata, packages_distributions
            dist = (packages_distributions().get(__package__) or [__package__.replace("_", "-")])[0]
            return metadata(dist)["Name"]
    
        @property
        def version(self) -> str:  # Tool version
    
    def version(self) -> str:  # Tool version
            from cjm_capability_ffmpeg import __version__
            return __version__
    
        def initialize(self,
                       config: Optional[Any] = None,  # Configuration dict or None for defaults
                      ) -> None
    
    def initialize(self,
                       config: Optional[Any] = None,  # Configuration dict or None for defaults
                      ) -> None
        "First-time setup. No storage init — the adapter owns the cache (stage 8)."
    
    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for UI form generation
            """Return the JSON Schema for tool configuration."""
            return dataclass_to_jsonschema(FFmpegCapabilityConfig)
    
        def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
        "Return the JSON Schema for tool configuration."
    
    def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
            """Return the current configuration as a dictionary."""
            return config_to_dict(self.config) if self.config else {}
    
        def is_available(self) -> bool:  # Whether ffmpeg is installed
        "Return the current configuration as a dictionary."
    
    def is_available(self) -> bool:  # Whether ffmpeg is installed
            """Check if ffmpeg is available on this system."""
            return FFMPEG_AVAILABLE
    
        # ------------------------------------------------------------------
        # Private helper
        # ------------------------------------------------------------------
    
        def _detect_audio_codec(self,
                                file_path: str,  # Path to media file
                               ) -> Optional[str]:  # Audio codec name or None
        "Check if ffmpeg is available on this system."
    
    def get_info(self,
                     file_path: Union[str, Path],  # Path to media file
                    ) -> MediaMetadata:  # Probed metadata
        "Probe metadata for a media file via ffprobe (UNCACHED pure-query)."
    
    def convert(self,
                    input_path: Union[str, Path],     # Source (or upstream artifact) media to convert
                    output_dir: str,                  # Adapter-chosen dir to write the artifact into
                    output_format: str,               # Target audio format (e.g. 'wav', 'mp3')
                    sample_rate: Optional[int] = None,  # Target sample rate (e.g. 16000); None = source rate
                    channels: Optional[int] = None,     # Target channel count (e.g. 1 = mono); None = source
                    **kwargs                          # Provenance pass-through (unused by compute)
                   ) -> MediaArtifactResult:  # The produced converted-audio artifact
        "Convert media to target/model-ready audio — PURE COMPUTE.

Writes `<output_dir>/<stem>.<output_format>` and returns its pointer. The
quality knobs (bitrate, resampler) come from `self.config`; the shape
params (format/rate/channels) are the per-call request (the adapter hashes
them into the cache key)."
    
    def segment_audio(self,
                          input_path: Union[str, Path],         # Source audio to cut
                          output_dir: str,                      # Adapter-chosen dir to write the segments into
                          boundaries: List[Dict[str, float]],   # [{"start", "end"}, ...]
                          output_format: Optional[str] = None,  # Output format (default: same as input)
                          filename_template: str = "segment_{index:03d}",  # Per-segment filename pattern
                          **kwargs                              # Provenance pass-through (unused by compute)
                         ) -> MediaSegmentationResult:  # The produced batch of segment files
        "Split an audio file into segments at the given boundaries — PURE COMPUTE.

Writes one file per boundary into `output_dir` and returns the typed batch
result. Batch-level caching is the adapter's concern (one row per
(input, boundaries+format config); the per-segment resume-skip the fused
tool did is intentionally not reproduced)."
    
    def extract_audio(self,
                          input_path: Union[str, Path],         # Source video container
                          output_dir: str,                      # Adapter-chosen dir to write the artifact into
                          output_format: Optional[str] = None,  # Audio format (default: from codec detection)
                          **kwargs                              # Provenance pass-through (unused by compute)
                         ) -> MediaArtifactResult:  # The produced extracted-audio artifact
        "Extract the audio stream from a video container — PURE COMPUTE.

Stream-copies when the source codec maps cleanly (lossless, fast),
otherwise re-encodes to the resolved format. Writes into `output_dir`."
```


### Audio Codec Map (`codec.ipynb`)
> Map audio container formats to the ffmpeg codec used to encode them.

#### Import

```python
from cjm_capability_ffmpeg.utils.codec import (
    get_audio_codec
)
```

#### Functions

```python
def get_audio_codec(audio_format: str  # The desired audio format (e.g. 'mp3', 'wav')
                   ) -> str:  # The ffmpeg audio codec name ('copy' if unknown)
    "Map an audio container format to the appropriate ffmpeg codec."
```


### Media Probing (`probe.ipynb`)
> Probe media files for metadata (duration, ...) via ffprobe.

#### Import

```python
from cjm_capability_ffmpeg.utils.probe import (
    get_media_duration
)
```

#### Functions

```python
def get_media_duration(file_path: Path  # Path to the media file
                      ) -> Optional[float]:  # Duration in seconds, or None if it cannot be determined
    "Get the duration of a media file (seconds) via ffprobe."
```


### FFmpeg Execution + Progress (`progress.ipynb`)
> Run ffmpeg subprocess commands with a progress bar and optional callback.

#### Import

```python
from cjm_capability_ffmpeg.utils.progress import (
    parse_progress_line,
    run_ffmpeg_with_progress
)
```

#### Functions

```python
def parse_progress_line(line: str  # A line of stderr output from ffmpeg
                        ) -> Optional[float]:  # Current time in seconds, or None if the line has no progress info
    "Parse a progress line from ffmpeg stderr output."
```

```python
def run_ffmpeg_with_progress(
    cmd: List[str],  # The ffmpeg command and arguments
    total_duration: Optional[float] = None,  # Total duration in seconds for a determinate bar, else indeterminate
    description: str = "Processing",  # Description text for the progress bar
    verbose: bool = False,  # If True, prints detailed ffmpeg output
    progress_callback: Optional[Callable[[float], None]] = None  # Optional callback receiving current progress in seconds
) -> None:  # Raises FileNotFoundError or subprocess.CalledProcessError on failure
    "Run an ffmpeg command with a progress bar."
```


### Audio Segment Extraction (`segments.ipynb`)
> Extract temporal segments from audio files via ffmpeg stream-copy.

#### Import

```python
from cjm_capability_ffmpeg.utils.segments import (
    extract_audio_segment
)
```

#### Functions

```python
def extract_audio_segment(input_path: Path,  # Path to the input audio file
                          output_path: Path,  # Path where the extracted segment is saved
                          start_time: str,  # Start time as "HH:MM:SS" or seconds
                          duration: str,  # Duration as "HH:MM:SS" or seconds
                          verbose: bool = False,  # If True, shows verbose ffmpeg output
                          pbar: bool = False,  # If True, shows a progress bar
                          copy_codec: bool = True,  # Stream-copy without re-encoding (fast)
                        ) -> None:  # Raises subprocess.CalledProcessError if extraction fails
    "Extract a temporal segment from an audio file."
```
